# Part 4 — Exploratory Data Analysis (EDA)
## Smart Healthcare Assistant and Hospital Management System
### Predicting AI-Assisted HMS Adoption Readiness in Bangladesh (Synthetic Dataset)

**Important notice:** This dataset (`synthetic_hms_adoption_dataset.csv`) is entirely **synthetic/simulated**. It does not contain real survey responses from Bangladeshi healthcare workers or hospitals. All findings in this notebook describe patterns **within the simulated sample only** and must not be interpreted as empirical evidence about real-world AI adoption in Bangladesh's healthcare sector.

This notebook performs **Exploratory Data Analysis only**. No model training, train/test splitting, imputation, scaling, or encoding is performed here — these steps belong to a later modeling stage.

## Section 1 — Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

df = pd.read_csv("synthetic_hms_adoption_dataset.csv")

print("First 5 rows:")
display(df.head())

print("\nShape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nBasic info:")
df.info()

## Section 2 — Dataset Overview

A concise overview of variable types. Detailed column-by-column descriptions were already covered in Part 2, so this section only summarizes counts and dtype groupings.

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"\nNumber of numerical variables: {len(numerical_cols)}")
print(f"Number of categorical variables: {len(categorical_cols)}")

print("\nNumerical variables:")
print(numerical_cols)

print("\nCategorical variables:")
print(categorical_cols)

## Section 3 — Descriptive Statistics

Descriptive statistics for the main numerical variables of interest.

In [ ]:
main_numeric_vars = [
    "age",
    "years_of_experience",
    "ai_awareness_score",
    "privacy_score",
    "human_factor_score",
    "infrastructure_score",
    "adoption_readiness_score",
]

desc_stats = df[main_numeric_vars].describe().T
desc_stats = desc_stats.rename(columns={
    "count": "Count", "mean": "Mean", "std": "Std Dev",
    "min": "Min", "25%": "25th Pct", "50%": "Median",
    "75%": "75th Pct", "max": "Max"
})
desc_stats.round(2)

**Interpretation (synthetic dataset):** The table above summarizes central tendency and spread for age, experience, and the four predictor/target scores within the simulated sample. Any skew, wide ranges, or unusual spread should be noted here based on the actual printed values before proceeding — these patterns reflect the synthetic data-generating process and are not claims about the real Bangladeshi healthcare workforce.

## Section 4 — Target Variable Analysis

Analyzing `adoption_readiness_category`, the main classification target for the later ML stage.

In [ ]:
category_order = ["Low", "Medium", "High"]

freq_table = df["adoption_readiness_category"].value_counts().reindex(category_order)
pct_table = (df["adoption_readiness_category"].value_counts(normalize=True) * 100).reindex(category_order)

target_summary = pd.DataFrame({
    "Count": freq_table,
    "Percentage (%)": pct_table.round(2)
})
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=target_summary.index, y=target_summary["Count"], order=category_order,
            palette="viridis", ax=ax)
ax.set_title("Distribution of Adoption Readiness Category (Synthetic Sample)")
ax.set_xlabel("Adoption Readiness Category")
ax.set_ylabel("Number of Respondents")
plt.tight_layout()
plt.show()

max_pct = pct_table.max()
min_pct = pct_table.min()
imbalance_ratio = freq_table.max() / freq_table.min()
print(f"Largest class share: {max_pct:.2f}%")
print(f"Smallest class share: {min_pct:.2f}%")
print(f"Imbalance ratio (largest/smallest): {imbalance_ratio:.2f}")

**Interpretation:** The counts and percentages above should be checked directly — do not assume balance. If the imbalance ratio is close to 1, classes are roughly balanced; a ratio noticeably above ~1.5–2 suggests meaningful class imbalance in the simulated sample. Class imbalance matters for the later ML stage because it can bias a classifier toward the majority class and inflate accuracy while undermining minority-class recall, so metrics such as macro-F1 or balanced accuracy will be more informative than raw accuracy in that stage.

## Section 5 — Distribution of Main Predictor Scores

Individual distributions for the four main predictor scores: AI awareness, privacy concern, human-factor resistance, and infrastructure readiness.

In [ ]:
predictor_scores = ["ai_awareness_score", "privacy_score", "human_factor_score", "infrastructure_score"]
titles = ["AI Awareness Score", "Privacy Concern Score", "Human-Factor Resistance Score", "Infrastructure Readiness Score"]

for col, title in zip(predictor_scores, titles):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.histplot(df[col], kde=True, bins=20, color="steelblue", ax=ax)
    ax.set_title(f"Distribution of {title} (Synthetic Sample)")
    ax.set_xlabel(title)
    ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.show()

**Interpretation:** Note the approximate center and shape (symmetric, skewed, multi-modal) of each distribution as printed above. These describe how the synthetic data-generating process assigned scores in the simulated sample and are not descriptions of real clinician or hospital populations.

## Section 6 — Predictor Scores vs Adoption Readiness

This section visually examines the relationship between each predictor score and adoption readiness, using both the categorical target (boxplots) and the continuous target (scatterplots with trend lines).

Expected directions (to be checked, not assumed):
- AI awareness → expected **positive** relationship
- Infrastructure readiness → expected **positive** relationship
- Privacy concern → expected **negative** relationship
- Human-factor resistance → expected **negative** relationship

In [ ]:
for col, title in zip(predictor_scores, titles):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    sns.boxplot(x="adoption_readiness_category", y=col, data=df,
                order=category_order, palette="Set2", ax=axes[0])
    axes[0].set_title(f"{title} by Adoption Readiness Category")
    axes[0].set_xlabel("Adoption Readiness Category")
    axes[0].set_ylabel(title)

    sns.regplot(x=col, y="adoption_readiness_score", data=df,
                scatter_kws={"alpha": 0.3, "s": 15}, line_kws={"color": "red"}, ax=axes[1])
    axes[1].set_title(f"{title} vs Adoption Readiness Score")
    axes[1].set_xlabel(title)
    axes[1].set_ylabel("Adoption Readiness Score")

    plt.tight_layout()
    plt.show()

**Interpretation:** Compare the observed direction and steepness of each trend line/boxplot pattern above against the expected direction stated earlier. Report only what is actually observed in the simulated sample — if a predictor does not show the expected relationship, state that plainly rather than forcing the interpretation to match expectations.

## Section 7 — Correlation Analysis

Pearson correlations among the four predictor scores and the continuous target, `adoption_readiness_score`.

In [ ]:
corr_vars = ["ai_awareness_score", "privacy_score", "human_factor_score",
             "infrastructure_score", "adoption_readiness_score"]

corr_matrix = df[corr_vars].corr(method="pearson")
display(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Heatmap: Predictor Scores and Adoption Readiness (Synthetic Sample)")
plt.tight_layout()
plt.show()

**Interpretation guide (report actual values, not assumed ones):**
- |r| < 0.20 → weak
- 0.20 ≤ |r| < 0.40 → moderate
- |r| ≥ 0.40 → strong

Using this guide, describe the direction and approximate strength of each predictor's correlation with `adoption_readiness_score` as printed above, e.g., "AI awareness has a [weak/moderate/strong] positive association with adoption readiness in the synthetic dataset." Correlation does not imply causation, and these coefficients only describe patterns within the simulated sample.

## Section 8 — Statistical Significance

Pearson correlation coefficients and p-values for each predictor score against `adoption_readiness_score`, using α = 0.05.

**Caution:** These p-values come from a synthetic dataset and must not be interpreted as evidence about the real Bangladeshi healthcare population. They only indicate whether a linear association is detectable within the simulated sample as generated.

In [ ]:
alpha = 0.05
sig_results = []

for col, title in zip(predictor_scores, titles):
    r, p = stats.pearsonr(df[col], df["adoption_readiness_score"])
    sig_results.append({
        "Predictor": title,
        "Pearson r": round(r, 3),
        "p-value": round(p, 5),
        "Significant at alpha=0.05": "Yes" if p < alpha else "No"
    })

sig_table = pd.DataFrame(sig_results)
display(sig_table)

## Section 9 — Profession Analysis

Comparing `adoption_readiness_score` across `profession` groups within the simulated sample.

In [ ]:
profession_summary = df.groupby("profession")["adoption_readiness_score"].agg(
    Count="count", Mean="mean", Std="std"
).round(2).sort_values("Mean", ascending=False)

display(profession_summary)

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.boxplot(x="profession", y="adoption_readiness_score", data=df,
            order=profession_summary.index, palette="pastel", ax=ax)
ax.set_title("Adoption Readiness Score by Profession (Synthetic Sample)")
ax.set_xlabel("Profession")
ax.set_ylabel("Adoption Readiness Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

**Interpretation:** Any differences in mean adoption readiness score across professions above describe a pattern **within the synthetic dataset only**. They must not be read as claims about which real profession (e.g., doctors vs. nurses vs. IT staff) is actually more ready to adopt AI-assisted HMS tools in practice.

## Section 10 — Hospital Type Analysis

Comparing `adoption_readiness_score` across `hospital_type` within the simulated sample.

In [ ]:
hospital_type_summary = df.groupby("hospital_type")["adoption_readiness_score"].agg(
    Count="count", Mean="mean", Std="std"
).round(2).sort_values("Mean", ascending=False)

display(hospital_type_summary)

fig, ax = plt.subplots(figsize=(7.5, 5))
sns.boxplot(x="hospital_type", y="adoption_readiness_score", data=df,
            order=hospital_type_summary.index, palette="muted", ax=ax)
ax.set_title("Adoption Readiness Score by Hospital Type (Synthetic Sample)")
ax.set_xlabel("Hospital Type")
ax.set_ylabel("Adoption Readiness Score")
plt.tight_layout()
plt.show()

**Interpretation:** Differences observed above reflect only the synthetic data-generating process, not real differences between public, private, or NGO hospitals in Bangladesh.

## Section 11 — Hospital Location Analysis

Comparing `adoption_readiness_score` across `hospital_location` within the simulated sample.

In [ ]:
location_summary = df.groupby("hospital_location")["adoption_readiness_score"].agg(
    Count="count", Mean="mean"
).round(2).sort_values("Mean", ascending=False)

display(location_summary)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(x="hospital_location", y="adoption_readiness_score", data=df,
            order=location_summary.index, palette="Set3", ax=ax)
ax.set_title("Adoption Readiness Score by Hospital Location (Synthetic Sample)")
ax.set_xlabel("Hospital Location")
ax.set_ylabel("Adoption Readiness Score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Section 12 — Missing Value Analysis

The dataset intentionally contains missing values in some Likert-item columns (the individual `_q#` questions), though the aggregated score columns are complete. Missing-value treatment (imputation) will be handled later inside the ML preprocessing pipeline — **no imputation or row removal is performed here.**

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": missing_pct
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values(
    "Missing Count", ascending=False
)
display(missing_summary)

if len(missing_summary) > 0:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    sns.barplot(x=missing_summary.index, y=missing_summary["Missing Percentage (%)"],
                color="salmon", ax=ax)
    ax.set_title("Missing Value Percentage by Column (Synthetic Sample)")
    ax.set_xlabel("Column")
    ax.set_ylabel("Missing (%)")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in the dataset.")

**Note:** Missing-value treatment will be handled later inside the ML preprocessing pipeline. This section only documents where and how much data is missing.

## Section 13 — Key EDA Findings

A concise, automatically generated summary of the most important findings, clearly labeled as describing the synthetic dataset only.

In [ ]:
target_dist_str = ", ".join([f"{cat}: {pct_table[cat]:.1f}%" for cat in category_order])
mean_adoption = df["adoption_readiness_score"].mean()

corr_lookup = {row["Predictor"]: row["Pearson r"] for row in sig_results}

top_profession = profession_summary["Mean"].idxmax()
bottom_profession = profession_summary["Mean"].idxmin()
top_hospital_type = hospital_type_summary["Mean"].idxmax()
bottom_hospital_type = hospital_type_summary["Mean"].idxmin()

print("=== KEY EDA FINDINGS (SYNTHETIC DATASET ONLY) ===\n")
print(f"1. Target distribution (adoption_readiness_category): {target_dist_str}")
print(f"2. Mean adoption readiness score: {mean_adoption:.2f}")
print(f"3. AI awareness correlation with adoption readiness (r): {corr_lookup['AI Awareness Score']}")
print(f"4. Privacy concern correlation with adoption readiness (r): {corr_lookup['Privacy Concern Score']}")
print(f"5. Human-factor correlation with adoption readiness (r): {corr_lookup['Human-Factor Resistance Score']}")
print(f"6. Infrastructure correlation with adoption readiness (r): {corr_lookup['Infrastructure Readiness Score']}")
print(f"7. Profession with highest simulated mean adoption readiness: {top_profession}")
print(f"8. Profession with lowest simulated mean adoption readiness: {bottom_profession}")
print(f"9. Hospital type with highest simulated mean adoption readiness: {top_hospital_type}")
print(f"10. Hospital type with lowest simulated mean adoption readiness: {bottom_hospital_type}")
print("\nAll findings above describe patterns within the synthetic/simulated sample only.")

## Section 14 — EDA Conclusion

**What did we learn from the synthetic dataset?**

- **Target distribution:** Review the printed class percentages in Section 4 to describe whether `adoption_readiness_category` is balanced or skewed toward one class within the simulated sample.
- **Strongest predictor relationships:** Based on Section 7–8, identify which of AI awareness, privacy concern, human-factor resistance, and infrastructure readiness shows the largest-magnitude correlation with `adoption_readiness_score`.
- **Direction of major relationships:** State whether each predictor moved in the expected direction (AI awareness and infrastructure positive; privacy concern and human-factor resistance negative) as actually observed, not assumed.
- **Missing-value situation:** Summarize which columns had missing values and their approximate percentages, noting that treatment is deferred to the preprocessing pipeline.
- **Class imbalance:** State whether the imbalance ratio computed in Section 4 indicates meaningful imbalance requiring attention (e.g., stratified splitting, class weighting) in the later ML stage.
- **Promising variables for ML:** Based on the correlation and group-difference results above, note which predictor scores appear most informative for later modeling.

**These observations describe the synthetic dataset and cannot be interpreted as empirical evidence about AI-assisted HMS adoption in Bangladesh.**